# Role 3: Graph Construction & Visualisation

This notebook:
1. Builds the heterogeneous FMA graph
2. Visualises the **genre subgraph** (which genres are most connected?)
3. Visualises an **ego graph** for a specific track
4. After training: plots **t-SNE of GNN embeddings** vs CLAP embeddings
5. Shows the **degree distribution** (most central tracks/artists)

Run `python -m src.graph.train` first to generate `gnn_embeddings.npy`.

In [ ]:
import sys
sys.path.insert(0, '..')  # make sure src/ is importable from notebooks/

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import networkx as nx
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from collections import Counter

from src.graph.build_graph import build_hetero_graph
from src.metadata import load_tracks, load_genres, get_small_subset_ids
from src.config import PROCESSED_DIR

print('Imports OK')

## 1. Build the Graph

In [ ]:
data, id_maps = build_hetero_graph(verbose=True)
print('\nHeteroData:')
print(data)

## 2. Genre Subgraph — Which genres are most connected?

We build a weighted undirected graph where:
- **Nodes** = genres
- **Edge weight** = number of tracks that share the same artist AND span both genres

In [ ]:
tracks = load_tracks()
genres_df = load_genres()
small_ids = get_small_subset_ids(tracks)
tracks_small = tracks.loc[tracks.index.isin(small_ids)]

# Build genre co-occurrence via shared artists
# For each artist, get all genres their tracks belong to
artist_genres = tracks_small.groupby(('artist', 'id'))[('track', 'genre_top')].apply(list)

genre_cooccur = Counter()
for genres_list in artist_genres:
    unique_genres = list(set(g for g in genres_list if isinstance(g, str)))
    for i in range(len(unique_genres)):
        for j in range(i + 1, len(unique_genres)):
            pair = tuple(sorted([unique_genres[i], unique_genres[j]]))
            genre_cooccur[pair] += 1

# Build networkx graph
G_genre = nx.Graph()
for (g1, g2), weight in genre_cooccur.items():
    if weight >= 2:  # filter low-weight edges for clarity
        G_genre.add_edge(g1, g2, weight=weight)

# Node sizes = number of tracks in that genre
genre_counts = tracks_small[('track', 'genre_top')].value_counts()
node_sizes = [genre_counts.get(g, 1) * 3 for g in G_genre.nodes()]

print(f'Genre subgraph: {G_genre.number_of_nodes()} nodes, {G_genre.number_of_edges()} edges')

In [ ]:
fig, ax = plt.subplots(figsize=(14, 10))

pos = nx.spring_layout(G_genre, seed=42, k=2.5)
weights = [G_genre[u][v]['weight'] for u, v in G_genre.edges()]
max_w = max(weights) if weights else 1

nx.draw_networkx_nodes(G_genre, pos, node_size=node_sizes, node_color='steelblue', alpha=0.85, ax=ax)
nx.draw_networkx_labels(G_genre, pos, font_size=9, font_color='white', font_weight='bold', ax=ax)
nx.draw_networkx_edges(G_genre, pos,
                       width=[2 + 6 * w / max_w for w in weights],
                       alpha=0.4, edge_color='gray', ax=ax)

ax.set_title('FMA Genre Subgraph\n(node size = track count, edge width = artist cross-genre connections)', fontsize=13)
ax.axis('off')
plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'gnn_genre_subgraph.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: gnn_genre_subgraph.png')

## 3. Ego Graph — Neighbourhood of a specific track

Pick a seed track and visualise its 2-hop neighbourhood: its artist, genre, and all tracks sharing that connectivity.

In [ ]:
# ---- Pick a seed track ----
# Change this to any track ID in the small subset
SEED_TRACK_ID = small_ids[0]

seed_info = tracks_small.loc[SEED_TRACK_ID]
seed_title  = seed_info[('track', 'title')]
seed_artist = seed_info[('artist', 'name')]
seed_genre  = seed_info[('track', 'genre_top')]
print(f'Seed track: [{SEED_TRACK_ID}] "{seed_title}" by {seed_artist} — {seed_genre}')

# Build a small networkx graph around this track (2-hop)
# Nodes: the track, its artist, its genre, sibling tracks (same genre)
seed_artist_id = seed_info[('artist', 'id')]

# Tracks by the same artist
same_artist_tracks = tracks_small[tracks_small[('artist', 'id')] == seed_artist_id].index.tolist()
# Tracks with the same top genre
same_genre_tracks  = tracks_small[tracks_small[('track', 'genre_top')] == seed_genre].index.tolist()

# Limit for readability
same_genre_tracks = same_genre_tracks[:15]

G_ego = nx.Graph()

# Add genre node
G_ego.add_node(f'GENRE:{seed_genre}', node_type='genre', label=seed_genre)
# Add artist node
G_ego.add_node(f'ARTIST:{seed_artist}', node_type='artist', label=seed_artist[:20])

# Add seed track
G_ego.add_node(f'TRACK:{SEED_TRACK_ID}', node_type='track_seed', label=str(SEED_TRACK_ID))
G_ego.add_edge(f'TRACK:{SEED_TRACK_ID}', f'ARTIST:{seed_artist}', rel='made_by')
G_ego.add_edge(f'TRACK:{SEED_TRACK_ID}', f'GENRE:{seed_genre}',   rel='tagged')

# Add sibling tracks (same artist)
for tid in same_artist_tracks:
    if tid == SEED_TRACK_ID: continue
    G_ego.add_node(f'TRACK:{tid}', node_type='track_artist', label=str(tid))
    G_ego.add_edge(f'TRACK:{tid}', f'ARTIST:{seed_artist}', rel='made_by')

# Add co-genre tracks
for tid in same_genre_tracks:
    if tid == SEED_TRACK_ID: continue
    if f'TRACK:{tid}' not in G_ego:
        G_ego.add_node(f'TRACK:{tid}', node_type='track_genre', label=str(tid))
    G_ego.add_edge(f'TRACK:{tid}', f'GENRE:{seed_genre}', rel='tagged')

print(f'Ego graph: {G_ego.number_of_nodes()} nodes, {G_ego.number_of_edges()} edges')

In [ ]:
color_map = {
    'track_seed':   '#E74C3C',   # red — seed track
    'track_artist': '#F39C12',   # orange — same artist
    'track_genre':  '#3498DB',   # blue — same genre
    'artist':       '#2ECC71',   # green — artist node
    'genre':        '#9B59B6',   # purple — genre node
}
size_map = {'track_seed': 600, 'track_artist': 300, 'track_genre': 200, 'artist': 500, 'genre': 500}

node_colors = [color_map[G_ego.nodes[n]['node_type']] for n in G_ego.nodes()]
node_sizes  = [size_map[G_ego.nodes[n]['node_type']]  for n in G_ego.nodes()]
labels      = {n: G_ego.nodes[n]['label'] for n in G_ego.nodes()}

fig, ax = plt.subplots(figsize=(13, 9))
pos = nx.spring_layout(G_ego, seed=7, k=1.8)

nx.draw_networkx_nodes(G_ego, pos, node_color=node_colors, node_size=node_sizes, alpha=0.9, ax=ax)
nx.draw_networkx_edges(G_ego, pos, alpha=0.35, edge_color='#555', ax=ax)
nx.draw_networkx_labels(G_ego, pos, labels=labels, font_size=7, ax=ax)

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#E74C3C', label=f'Seed track ({SEED_TRACK_ID})'),
    Patch(facecolor='#F39C12', label='Same artist'),
    Patch(facecolor='#3498DB', label='Same genre'),
    Patch(facecolor='#2ECC71', label='Artist node'),
    Patch(facecolor='#9B59B6', label='Genre node'),
]
ax.legend(handles=legend_elements, loc='upper left', fontsize=9)
ax.set_title(f'Ego Graph: "{seed_title}" by {seed_artist}\n(2-hop neighbourhood)', fontsize=12)
ax.axis('off')
plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'gnn_ego_graph.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: gnn_ego_graph.png')

## 4. t-SNE of GNN Embeddings (run AFTER training)

Requires `data/processed/gnn_embeddings.npy` — run `python -m src.graph.train` first.

In [ ]:
gnn_embs_path = PROCESSED_DIR / 'gnn_embeddings.npy'
gnn_ids_path  = PROCESSED_DIR / 'gnn_track_ids.npy'

if not gnn_embs_path.exists():
    print('GNN embeddings not found. Run: python -m src.graph.train')
else:
    gnn_embs     = np.load(gnn_embs_path)
    gnn_track_ids = np.load(gnn_ids_path).tolist()

    # Get genre labels for colouring
    genre_labels = []
    for tid in gnn_track_ids:
        try:
            g = tracks_small.loc[tid, ('track', 'genre_top')]
            genre_labels.append(g if isinstance(g, str) else 'Unknown')
        except KeyError:
            genre_labels.append('Unknown')

    # PCA pre-reduction then t-SNE
    print(f'GNN embeddings shape: {gnn_embs.shape}')
    print('Running PCA (50 components)...')
    pca = PCA(n_components=min(50, gnn_embs.shape[1]), random_state=42)
    embs_pca = pca.fit_transform(gnn_embs)

    print('Running t-SNE...')
    tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
    embs_2d = tsne.fit_transform(embs_pca)
    print('Done.')

In [ ]:
if gnn_embs_path.exists():
    # Load CLAP t-SNE for comparison
    clap_tsne_path = PROCESSED_DIR / 'tsne_clap_embeddings.png'

    unique_genres = sorted(set(genre_labels))
    cmap = plt.colormaps['tab20'].resampled(len(unique_genres))
    genre_to_color = {g: cmap(i) for i, g in enumerate(unique_genres)}
    colors = [genre_to_color[g] for g in genre_labels]

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))

    # Left: GNN t-SNE
    ax = axes[0]
    sc = ax.scatter(embs_2d[:, 0], embs_2d[:, 1], c=colors, s=4, alpha=0.6)
    ax.set_title('GNN Embeddings — t-SNE\n(coloured by genre)', fontsize=12)
    ax.set_xlabel('t-SNE dim 1')
    ax.set_ylabel('t-SNE dim 2')
    ax.axis('off')

    # Legend
    from matplotlib.lines import Line2D
    legend_elems = [Line2D([0], [0], marker='o', color='w', markerfacecolor=genre_to_color[g],
                           markersize=7, label=g) for g in unique_genres[:16]]
    ax.legend(handles=legend_elems, loc='lower right', fontsize=7, ncol=2)

    # Right: CLAP t-SNE (image)
    if clap_tsne_path.exists():
        img = plt.imread(clap_tsne_path)
        axes[1].imshow(img)
        axes[1].set_title('CLAP Embeddings — t-SNE (reference)', fontsize=12)
        axes[1].axis('off')
    else:
        axes[1].text(0.5, 0.5, 'CLAP t-SNE not found\n(run notebook 03)', ha='center', va='center')

    plt.suptitle('GNN vs CLAP Embedding Space — Genre Clustering Comparison', fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig(PROCESSED_DIR / 'gnn_tsne_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: gnn_tsne_comparison.png')

## 5. Degree Distribution — Most Central Tracks & Artists

In [ ]:
# Track degree in the co-genre graph (how many other tracks share your genre?)
tg_edges = data['track', 'tagged', 'genre'].edge_index
track_genre_degree = Counter(tg_edges[0].tolist())  # how many genre tags per track

# Artist degree (how many tracks?)
ta_edges = data['track', 'made_by', 'artist'].edge_index
artist_degree = Counter(ta_edges[1].tolist())

# Top 10 most productive artists
top_artists = artist_degree.most_common(10)
print('Top 10 most-connected artists (by track count):')
for artist_idx, count in top_artists:
    print(f'  Artist node {artist_idx}: {count} tracks')

# Plot degree histogram
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

artist_counts_vals = list(artist_degree.values())
axes[0].hist(artist_counts_vals, bins=40, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Number of tracks per artist')
axes[0].set_ylabel('Number of artists')
axes[0].set_title('Artist Degree Distribution')
axes[0].set_yscale('log')

track_degree_vals = list(track_genre_degree.values())
axes[1].hist(track_degree_vals, bins=10, color='darkorange', edgecolor='white')
axes[1].set_xlabel('Number of genre tags per track')
axes[1].set_ylabel('Number of tracks')
axes[1].set_title('Track-Genre Edge Distribution')

plt.tight_layout()
plt.savefig(PROCESSED_DIR / 'gnn_degree_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: gnn_degree_distribution.png')